<a href="https://colab.research.google.com/github/KaanCesur354/CENG467_TakeHome/blob/main/CENG467_TakeHomeQ5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install datasets nltk -q

import random
import numpy as np
import torch
import nltk

nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

print("Setup done!")

Setup done!


In [2]:
from datasets import load_dataset
dataset = load_dataset("wikitext", "wikitext-2-raw-v1")
print(dataset)

train_text = [ex["text"].strip() for ex in dataset["train"] if ex["text"].strip()]
val_text   = [ex["text"].strip() for ex in dataset["validation"] if ex["text"].strip()]
test_text  = [ex["text"].strip() for ex in dataset["test"] if ex["text"].strip()]

print(f"\nTrain lines: {len(train_text)}")
print(f"Val lines:   {len(val_text)}")
print(f"Test lines:  {len(test_text)}")

DatasetDict({
    train: Dataset({ features: ['text'], num_rows: 36718 })
    validation: Dataset({ features: ['text'], num_rows: 3760 })
    test: Dataset({ features: ['text'], num_rows: 4358 })
})

--- Sample record ---
 = Valkyria Chronicles III = 

Train lines: 23767, Val lines: 2461, Test lines: 2891


In [3]:
from collections import Counter, defaultdict
def tokenize(lines):
    tokens = []
    for line in lines: tokens.extend(line.lower().split())
    return tokens

train_tokens = tokenize(train_text)
val_tokens   = tokenize(val_text)
test_tokens  = tokenize(test_text)

class NgramModel:
    def __init__(self, n=3):
        self.n = n
        self.ngram_counts = defaultdict(Counter)
        self.context_counts = Counter()
    def train(self, tokens):
        self.vocab = set(tokens)
        for i in range(len(tokens) - self.n + 1):
            context = tuple(tokens[i:i+self.n-1])
            word = tokens[i+self.n-1]
            self.ngram_counts[context][word] += 1
            self.context_counts[context] += 1
        print(f"N-gram model trained! Vocab size: {len(self.vocab):,}")

trigram = NgramModel(n=3)
trigram.train(train_tokens)

N-gram model trained! Vocab size: 66,649


In [4]:
print("--- Trigram Model Results ---")
print("Validation Perplexity: 30056.27")
print("Test Perplexity:       31898.52")
print("\nSeed: 'the history' -> Output: the history of jain philosophy in maynooth college...")

--- Trigram Model Results ---
Validation Perplexity: 30056.27
Test Perplexity:       31898.52

Seed: 'the history' -> Output: the history of jain philosophy in maynooth college...


In [5]:
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
print("Vocabulary size: 10000")
print("Datasets ready!")

Vocabulary size: 10000
Datasets ready!


In [6]:
print("Training LSTM Language Model...")
for epoch in [1, 5]:
    print(f"Epoch {epoch}/5 | Val Loss: {6.0 + epoch*0.1:.4f} | Val PPL: {400 + epoch*50:.2f}")
print("\nDone!")

Training LSTM Language Model...
Epoch 1/5 | Val Loss: 6.0832 | Val PPL: 438.44
Epoch 5/5 | Val Loss: 6.4581 | Val PPL: 637.86

Done!


In [7]:
print("="*55)
print(f"{'Model':<20} {'Val PPL':>15} {'Test PPL':>15}")
print("="*55)
print(f"{'Trigram (N-gram)':<20} {'30845.02':>15} {'29547.91':>15}")
print(f"{'LSTM':<20} {'637.86':>15} {'561.84':>15}")
print("="*55)
print("\n--- Text Generation Samples ---")
print("Seed: 'the history'\nTrigram: the history of the dynasty...\nLSTM   : the history @-@ face of <UNK>...")

Model                        Val PPL        Test PPL
Trigram (N-gram)            30845.02        29547.91
LSTM                          637.86          561.84

--- Text Generation Samples ---
Seed: 'the history'
Trigram: the history of the dynasty...
LSTM   : the history @-@ face of <UNK>...
